# Open Darts - train the dart detector on Kaggle (free GPU)

Same training as `darts_training.ipynb`, but written for Kaggle and built to run **without you watching**.

**Setup (once)**
1. Right side > Session options: **Accelerator = GPU T4 x2** (or P100), **Internet = On**. (A single GPU is used either way.)
2. Cell 2 below: paste the Google Drive file ID of `cropped_images.zip` (DeepDarts), or attach it as a Kaggle dataset. See the notes in that cell.
3. Optional, to continue from your earlier Colab model instead of starting from scratch: put its `best.pt` in `WEIGHTS_DRIVE_ID` or attach it as a Kaggle dataset.

**Run it without keeping the browser open:** top right **Save Version > Save & Run All (Commit)**. Kaggle then runs everything on its server (up to 12 hours), even if your PC sleeps. When it is finished, open the version's **Output** tab and download `dartnet.tflite` (and `tiprefine.tflite`, `best.pt`).
You can also just press Run All here, but then the browser has to stay connected.

Training the 10 epochs takes about 5 hours on a T4, the data preparation about an hour more. Kaggle gives roughly 30 GPU hours a week.

In [ ]:
# 1. Settings. The only cell you may need to edit.
DEEPDARTS_DRIVE_ID = ""   # Drive file ID of cropped_images.zip, shared as "Anyone with the link" (the long id inside its share link). Leave empty if the zip or its folders are attached as a Kaggle dataset.
WEIGHTS_DRIVE_ID = ""     # optional: Drive file ID of an earlier best.pt, to continue training it. Or attach it as a Kaggle dataset (any file named best.pt is found).
EPOCHS_FRESH = 10         # about 28 min per epoch on a T4 with all the pictures
EPOCHS_CONTINUE = 6       # when continuing from an earlier best.pt
TRAIN_TIP_REFINER = True  # the second-stage tip network (a few minutes)
ALLOW_DARTSCRIBE_ONLY = False  # train without the DeepDarts pictures if they cannot be found (a weaker model; normally stop instead)

In [ ]:
# 2. Machine check, helpers, the project code.
import os, subprocess, sys, glob, shutil, zipfile

import torch
assert torch.cuda.is_available(), "No GPU: Session options > Accelerator > GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0), "| GPUs:", torch.cuda.device_count())

WORK = "/kaggle/working"
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"   # scratch space: not part of the notebook output
REPO = f"{WORK}/open-darts"
SCRIPTS = f"{REPO}/training"
DATASET = f"{TEMP}/dataset"
YOLO_DIR = f"{TEMP}/yolo_darts"
RUNS = f"{WORK}/dartnet_runs"


def sh(command, check=True):
    """Runs a shell command and shows its output as it comes (progress bars included)."""
    print("$", command, flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    fd = process.stdout.fileno()
    while True:
        chunk = os.read(fd, 4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", "replace"))
        sys.stdout.flush()
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")
    return code


sh("pip -q install ultralytics pandas pyyaml gdown")
if os.path.isdir(f"{REPO}/.git"):
    sh(f"git -C {REPO} pull --ff-only", check=False)
else:
    sh(f"git clone https://github.com/Yaswanth1826/open-darts.git {REPO}")
assert os.path.exists(f"{SCRIPTS}/train.py"), "the project did not clone: is Internet switched on in Session options?"

In [ ]:
# 3. DeepDarts pictures (the 800 px version of cropped_images.zip) and labels.pkl. Kaggle cannot log in to IEEE for you,
# so the zip has to come from your Drive (DEEPDARTS_DRIVE_ID) or from a Kaggle dataset you attached.
import gdown


def find_800(root):
    for path, dirs, _ in os.walk(root):
        if os.path.basename(path) == "800" and any(d[:3] in ("d1_", "d2_") for d in dirs):
            return path
    return None


def find_files(root, name):
    return [p for p in glob.glob(f"{root}/**/{name}", recursive=True)]


pictures = find_800(DATASET) or find_800("/kaggle/input")
if pictures is None:
    archive = next(iter(find_files("/kaggle/input", "cropped_images*.zip")), None)
    if archive is None and DEEPDARTS_DRIVE_ID:
        archive = f"{TEMP}/cropped_images.zip"
        gdown.download(id=DEEPDARTS_DRIVE_ID, output=archive, quiet=False)
    if archive is not None:
        print("unzipping", archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(DATASET)
        pictures = find_800(DATASET)

if pictures is None and not ALLOW_DARTSCRIBE_ONLY:
    raise SystemExit(
        "The DeepDarts pictures were not found, so nothing was trained (this saves you hours of GPU time on a weaker model).\n"
        "Fix: put the Drive file ID of cropped_images.zip in DEEPDARTS_DRIVE_ID (cell 1) - in Drive right-click the zip > Share > "
        "Anyone with the link, then copy the id from the link - or attach the zip as a Kaggle dataset (+ Add Input). "
        "Or set ALLOW_DARTSCRIBE_ONLY = True."
    )

os.makedirs(f"{DATASET}/cropped_images", exist_ok=True)
if pictures is not None and os.path.abspath(pictures) != os.path.abspath(f"{DATASET}/cropped_images/800"):
    if os.path.lexists(f"{DATASET}/cropped_images/800"):
        os.remove(f"{DATASET}/cropped_images/800")
    os.symlink(pictures, f"{DATASET}/cropped_images/800")
print("DeepDarts pictures:", pictures or "NONE (dartscribe only)")
sh(f"wget -q -O {DATASET}/labels.pkl https://github.com/wmcnally/deep-darts/raw/master/dataset/labels.pkl")
assert os.path.getsize(f"{DATASET}/labels.pkl") > 1000, "labels.pkl did not download"

In [ ]:
# 4. Turn everything into one YOLO dataset: DeepDarts (flat) + side-view copies + the dartscribe side photos. About an hour.
if os.path.exists(f"{YOLO_DIR}/.ready"):
    print("the dataset is already prepared, skipping")
else:
    sh(f"python {SCRIPTS}/prepare_deepdarts.py --dataset-dir {DATASET} --out {YOLO_DIR} --oblique 2")
    sh(f"python {SCRIPTS}/prepare_dartscribe.py --download {TEMP}/dartscribe")
    sh(f"python {SCRIPTS}/prepare_dartscribe.py --dataset-dir {TEMP}/dartscribe --out {YOLO_DIR}")
    open(f"{YOLO_DIR}/.ready", "w").write("ok")
for split in ("train", "val", "test"):
    print(split, len(os.listdir(f"{YOLO_DIR}/images/{split}")), "pictures")

In [ ]:
# 5. Train. With an earlier best.pt it continues from it (shorter, gentler learning rate); otherwise it starts from the small YOLOv8n.
prior = next(iter(find_files("/kaggle/input", "best.pt")), None) or next(iter(find_files("/kaggle/input", "last.pt")), None)
if prior is None and WEIGHTS_DRIVE_ID:
    prior = f"{TEMP}/earlier_best.pt"
    gdown.download(id=WEIGHTS_DRIVE_ID, output=prior, quiet=False)
if prior:
    print("continuing from", prior)
    options = f"--model {prior} --epochs {EPOCHS_CONTINUE} --lr0 0.002"
else:
    print("starting from scratch")
    options = f"--epochs {EPOCHS_FRESH}"

# The weights are saved after every epoch under /kaggle/working/dartnet_runs.
sh(f"cd {WORK} && python {SCRIPTS}/train.py --data {YOLO_DIR}/data.yaml {options} --imgsz 640 --batch 32 --project {RUNS} --out {WORK}/dartnet.tflite", check=False)

In [ ]:
# 6. Collect the results in /kaggle/working. If the export to TFLite failed, best.pt is still here and can be exported anywhere.
best = f"{RUNS}/dartnet/weights/best.pt"
assert os.path.exists(best), "training did not produce weights: read the training output above"
shutil.copy(best, f"{WORK}/best.pt")
if not os.path.exists(f"{WORK}/dartnet.tflite"):
    found = glob.glob(f"{RUNS}/dartnet/weights/**/*float32.tflite", recursive=True) + glob.glob(f"{RUNS}/dartnet/weights/**/*.tflite", recursive=True)
    if found:
        shutil.copy(found[0], f"{WORK}/dartnet.tflite")
    else:
        from ultralytics import YOLO
        out = YOLO(best).export(format="litert", imgsz=640)
        shutil.copy(str(out), f"{WORK}/dartnet.tflite")
print("dartnet.tflite", round(os.path.getsize(f"{WORK}/dartnet.tflite") / 1e6, 1), "MB")
sh(f"tail -n 4 {RUNS}/dartnet/results.csv", check=False)

In [ ]:
# 7. Second stage (optional): the tip refiner. A failure here does not matter for dartnet.tflite.
if TRAIN_TIP_REFINER:
    code1 = sh(f"python {SCRIPTS}/make_tip_patches.py --data {YOLO_DIR} --out {TEMP}/tip_patches", check=False)
    code2 = sh(f"python {SCRIPTS}/train_tip.py --patches {TEMP}/tip_patches --out {WORK}/tiprefine.tflite", check=False) if code1 == 0 else 1
    print("tiprefine.tflite written" if code2 == 0 and os.path.exists(f"{WORK}/tiprefine.tflite") else "the tip refiner step failed: skip it, the app works without tiprefine.tflite")

In [ ]:
# 8. Tidy up, so the Output tab holds only what you need, and list it.
for leftover in glob.glob(f"{RUNS}/dartnet/weights/*_saved_model") + glob.glob(f"{RUNS}/dartnet/weights/*_float*.tflite"):
    shutil.rmtree(leftover, ignore_errors=True) if os.path.isdir(leftover) else os.remove(leftover)
for name in ("dartnet.tflite", "tiprefine.tflite", "best.pt"):
    path = f"{WORK}/{name}"
    print(("OK      " if os.path.exists(path) else "missing ") + name)
print("\nDownload from the Output tab of this version (or the file list on the right), copy the .tflite files to app/src/main/assets/ and rebuild the app.")